In [21]:
import os
import hashlib
import sqlite3
import time
from pathlib import Path
import pymupdf
from tqdm import tqdm
from dotenv import load_dotenv
from google import genai

# ==============================================================================
# KONFIGURASI UMUM (PATH ABSOLUT AGAR AMAN DI SEMUA KERNEL)
# ==============================================================================
BASE_DIR = Path(r"D:\Project\PDF Extraction")
if not BASE_DIR.exists():
    BASE_DIR = Path(os.getcwd())

PDF_SOURCE_DIR = str(Path(r"C:\Users\Arya\Downloads\Contoh pdf"))
DB_PATH = str(BASE_DIR / "data" / "rag_metadata.db")

CHUNK_SIZE = 800         # Karakter per potongan (~150 kata)
CHUNK_OVERLAP = 150      # Potongan yang saling bertumpuk agar konteks tidak putus
MIN_CHUNK_CHARS = 50     # Abaikan teks pendek seperti header/footer kosong
BATCH_COMMIT_SIZE = 5000 # Simpan ke disk sekaligus per 5.000 chunk

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
load_dotenv(dotenv_path=BASE_DIR / ".env")
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY belum dikonfigurasi di file .env!")

client = genai.Client(api_key=api_key)
print("Konfigurasi selesai dan siap dijalankan.")
print(f"Path Database: {DB_PATH} (File ada: {os.path.exists(DB_PATH)})")

Konfigurasi selesai dan siap dijalankan.


In [22]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Tuning performa SQLite untuk menangani jutaan baris data
cursor.execute("PRAGMA journal_mode = WAL;")        # Write-Ahead Logging (sangat cepat & aman konkurensi)
cursor.execute("PRAGMA synchronous = NORMAL;")      # Mengurangi hambatan I/O disk
cursor.execute("PRAGMA cache_size = -128000;")      # Alokasi memori cache ~128 MB RAM
cursor.execute("PRAGMA temp_store = MEMORY;")

# Tabel daftar dokumen (untuk checkpoint)
cursor.execute("""
CREATE TABLE IF NOT EXISTS documents (
    doc_id INTEGER PRIMARY KEY AUTOINCREMENT,
    file_name TEXT,
    file_path TEXT UNIQUE,
    checksum TEXT,
    total_pages INTEGER,
    status TEXT
)
""")

# Tabel chunk terstruktur
cursor.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    chunk_id INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id INTEGER,
    file_name TEXT,
    page_number INTEGER,
    content TEXT,
    FOREIGN KEY(doc_id) REFERENCES documents(doc_id)
)
""")

# Virtual table FTS5 untuk pencarian full-text BM25 instan
cursor.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS fts_chunks USING fts5(
    chunk_id UNINDEXED,
    file_name,
    page_number UNINDEXED,
    content
)
""")

conn.commit()
print("Struktur database SQLite & FTS5 siap.")

Struktur database SQLite & FTS5 siap.


In [23]:
def get_checksum(filepath, chunk_size=65536):
    hasher = hashlib.md5()
    try:
        with open(filepath, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                hasher.update(chunk)
        return hasher.hexdigest()
    except Exception:
        return ""

def split_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        if end < text_len:
            last_space = chunk.rfind(" ")
            if last_space > chunk_size * 0.7:
                end = start + last_space
                chunk = text[start:end]
        chunk_clean = chunk.strip()
        if len(chunk_clean) >= MIN_CHUNK_CHARS:
            chunks.append(chunk_clean)
        start = end - overlap
        if start >= text_len - overlap:
            break
    return chunks

# Ambil daftar file yang sudah selesai (Checkpoint)
cursor.execute("SELECT file_path FROM documents WHERE status = 'Indexed'")
processed_files = {row[0] for row in cursor.fetchall()}

pdf_files = list(Path(PDF_SOURCE_DIR).rglob("*.pdf"))
print(f"Total file PDF ditemukan : {len(pdf_files)}")
print(f"File sudah terindeks    : {len(processed_files)}")

chunk_buffer = []

def flush_buffer(buffer):
    if not buffer:
        return
    cursor.execute("SELECT COALESCE(MAX(chunk_id), 0) FROM chunks")
    max_id_before = cursor.fetchone()[0]

    cursor.executemany("""
    INSERT INTO chunks (doc_id, file_name, page_number, content)
    VALUES (?, ?, ?, ?)
    """, buffer)
    
    cursor.execute("""
    INSERT INTO fts_chunks (chunk_id, file_name, page_number, content)
    SELECT chunk_id, file_name, page_number, content
    FROM chunks
    WHERE chunk_id > ?
    """, (max_id_before,))
    
    conn.commit()
    buffer.clear()

# Eksekusi streaming per file dan per halaman
for pdf_path in tqdm(pdf_files, desc="Memproses PDF"):
    str_path = str(pdf_path)
    if str_path in processed_files:
        continue

    file_name = pdf_path.name
    checksum = get_checksum(str_path)

    try:
        doc = pymupdf.open(str_path)
        total_pages = len(doc)

        cursor.execute("""
        INSERT INTO documents (file_name, file_path, checksum, total_pages, status)
        VALUES (?, ?, ?, ?, 'Processing')
        ON CONFLICT(file_path) DO UPDATE SET status = 'Processing'
        """, (file_name, str_path, checksum, total_pages))
        doc_id = cursor.lastrowid

        # Streaming per lembar halaman (RAM tetap ringan)
        for page_idx in range(total_pages):
            page = doc.load_page(page_idx)
            text = page.get_text("text").strip()
            if not text:
                continue

            page_num = page_idx + 1
            page_chunks = split_text(text)

            for chunk in page_chunks:
                chunk_buffer.append((doc_id, file_name, page_num, chunk))
                if len(chunk_buffer) >= BATCH_COMMIT_SIZE:
                    flush_buffer(chunk_buffer)

        doc.close()
        
        cursor.execute("UPDATE documents SET status = 'Indexed' WHERE doc_id = ?", (doc_id,))
        conn.commit()

    except Exception as e:
        print(f"\n[Gagal] {file_name}: {e}")
        cursor.execute("""
        INSERT INTO documents (file_name, file_path, checksum, total_pages, status)
        VALUES (?, ?, ?, 0, 'Error')
        ON CONFLICT(file_path) DO UPDATE SET status = 'Error'
        """, (file_name, str_path, checksum))
        conn.commit()

# Simpan sisa potongan di buffer
flush_buffer(chunk_buffer)

cursor.execute("SELECT COUNT(*) FROM chunks")
total_chunks = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM documents WHERE status = 'Indexed'")
total_docs = cursor.fetchone()[0]

print("\n=== INDEXING SELESAI ===")
print(f"Total PDF berhasil diindeks : {total_docs}")
print(f"Total chunk dalam database  : {total_chunks}")
conn.close()

Total file PDF ditemukan : 31
File sudah terindeks    : 31


Memproses PDF: 100%|██████████| 31/31 [00:00<00:00, 445285.70it/s]


=== INDEXING SELESAI ===
Total PDF berhasil diindeks : 31
Total chunk dalam database  : 2292


In [24]:
INDONESIAN_STOPWORDS = {
    "apa", "apakah", "siapa", "mengapa", "kenapa", "bagaimana", "dimana",
    "kapan", "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk",
    "pada", "adalah", "sebagai", "dengan", "atau", "oleh", "dalam", "bisa",
    "dapat", "jelaskan", "sebutkan", "tentang", "tersebut", "ada"
}

def clean_and_build_fts_query(user_query: str) -> str:
    raw_tokens = "".join([c if c.isalnum() or c.isspace() else " " for c in user_query.lower()]).split()
    meaningful = [t for t in raw_tokens if t not in INDONESIAN_STOPWORDS and len(t) > 2]
    if not meaningful:
        meaningful = [t for t in raw_tokens if len(t) > 2]
    return " OR ".join(meaningful) if meaningful else ""

def retrieve_relevant_chunks(query: str, top_k: int = 8):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    fts_query = clean_and_build_fts_query(query)
    if not fts_query:
        conn.close()
        return []
        
    sql = """
    SELECT c.file_name, c.page_number, c.content, bm25(fts_chunks) AS rank
    FROM fts_chunks f
    JOIN chunks c ON f.chunk_id = c.chunk_id
    WHERE fts_chunks MATCH ?
    ORDER BY rank
    LIMIT ?
    """
    
    try:
        cursor.execute(sql, (fts_query, top_k))
        rows = cursor.fetchall()
    except Exception as e:
        print(f"[DEBUG SQL ERROR]: {e}")
        rows = []
    finally:
        conn.close()
        
    results = []
    for row in rows:
        results.append({
            "file_name": row[0],
            "page_number": row[1],
            "text": row[2].strip(),
            "score": round(row[3], 4)
        })
    return results

def ask_gemini_rag(query: str, top_k: int = 8, max_retries: int = 3):
    chunks = retrieve_relevant_chunks(query, top_k=top_k)
    
    if not chunks:
        return "Tidak ditemukan potongan teks yang relevan di database lokal untuk kata kunci tersebut.", []
        
    context_blocks = []
    for idx, c in enumerate(chunks, 1):
        context_blocks.append(
            f"[Sumber {idx}] Dokumen: {c['file_name']} (Halaman: {c['page_number']})\n{c['text']}"
        )
    context_str = "\n\n".join(context_blocks)
    
    prompt = f"""Kamu adalah analis dokumen akademik dan medis yang presisi.
Jawab pertanyaan berikut HANYA berdasarkan KONTEKS DOKUMEN yang tersedia di bawah ini.

Pedoman Jawaban:
1. Utamakan ketepatan faktual. Jangan berasumsi atau mengarang data di luar teks.
2. Jelaskan fakta penting secara runut, padat, dan sertakan rujukan nama file serta nomor halaman.
3. Jika dokumen hanya memuat sebagian informasi, jelaskan fakta yang ada terlebih dahulu dan sebutkan keterbatasannya secara jujur.

KONTEKS DOKUMEN:
{context_str}

PERTANYAAN:
{query}

JAWABAN ANALITIS:"""

    models_to_try = [
        "gemini-flash-latest",
        "gemini-3.5-flash",
        "gemini-3.5-flash-lite",
        "gemini-3.8-flash",
    ]

    for attempt, model_name in enumerate(models_to_try, 1):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            return response.text.strip(), chunks
        except Exception as e:
            time.sleep(2)
            
    return "Gagal mendapatkan respon dari Gemini API setelah beberapa percobaan.", chunks

# Input satu kali per eksekusi cell
user_query = input("Masukkan pertanyaan Anda: ").strip()

if not user_query:
    print("Pertanyaan tidak boleh kosong.")
else:
    print(f"\nMencari dokumen terkait: '{user_query}'...")
    jawaban, sources = ask_gemini_rag(user_query, top_k=8)
    
    print("\n" + "=" * 60)
    print("JAWABAN AI:")
    print("=" * 60)
    print(jawaban)
    
    if sources:
        print("\n" + "-" * 60)
        print("REFERENSI DOKUMEN YANG DIGUNAKAN:")
        print("-" * 60)
        for idx, s in enumerate(sources[:5], 1):
            print(f"[{idx}] {s['file_name']} (Halaman {s['page_number']})")


Mencari dokumen terkait: 'anda tau preeklmapsia?'...

JAWABAN AI:
Berdasarkan konteks dokumen yang disediakan, informasi mengenai **preeklampsia** (atau ditulis *preeklmapsia*) hanya dijelaskan secara terbatas pada aspek berikut:

1. **Waktu Terjadinya:** Preeklampsia dapat timbul pada masa kehamilan (*pregnancy*), persalinan (*labor*), maupun masa nifas (*postpartum*) (**[Sumber 1] `Terapi_Murottal_Sebagai_Upaya_Menurunkan.pdf`, Halaman: 3**).
2. **Keterkaitan Kondisi:** Dalam konteks kajian literatur dokumen tersebut, istilah preeklampsia dikaitkan atau dipadankan dengan kata kunci **tekanan darah / hipertensi** serta **kecemasan** (**[Sumber 1] `Terapi_Murottal_Sebagai_Upaya_Menurunkan.pdf`, Halaman: 3**).

**Keterbatasan:**
Dokumen yang tersedia tidak memuat definisi medis lengkap, patofisiologi, gejala khusus, komplikasi, maupun tata laksana klinis dari preeklampsia.

------------------------------------------------------------
REFERENSI DOKUMEN YANG DIGUNAKAN:
------------------